In [ ]:
"""
Restaurant Lead Scraper
-----------------------
Reads restaurant names from businesses.csv (OSM export),
searches each via Serper API, and extracts:
  - Official website
  - Phone number
  - Zomato link
  - Swiggy link
  - Google Maps link
  - Any other delivery/referral links

Output → results.csv
"""

import csv
import json
import os
import re
import time
import requests
from pathlib import Path

# ── Config ──────────────────────────────────────────────────────────────────
SERPER_API_KEY = "5aaf23f2e9f016e38bb99da0f136d5f9f66e0f58"
INPUT_FILE     = "businesses.csv"
OUTPUT_FILE    = "results.csv"
DELAY_SECONDS  = 1.2   # polite delay between API calls
CITY_HINT      = "Bhopal"  # appended to searches to narrow results (edit as needed)

REFERRAL_DOMAINS = {
    "zomato.com":       "zomato",
    "swiggy.com":       "swiggy",
    "magicpin.in":      "magicpin",
    "dineout.co.in":    "dineout",
    "eazydiner.com":    "eazydiner",
    "justdial.com":     "justdial",
    "yelp.com":         "yelp",
    "tripadvisor.com":  "tripadvisor",
}

OUTPUT_FIELDS = [
    "name", "city_hint",
    "official_website", "phone",
    "google_maps",
    "zomato", "swiggy", "magicpin",
    "dineout", "eazydiner", "justdial",
    "tripadvisor", "other_links",
    "search_title", "search_snippet",
]

# ── Helpers ──────────────────────────────────────────────────────────────────

def serper_search(query: str) -> dict:
    """Call Serper /search and return raw JSON."""
    resp = requests.post(
        "https://google.serper.dev/search",
        headers={
            "X-API-KEY": SERPER_API_KEY,
            "Content-Type": "application/json",
        },
        json={"q": query, "num": 10, "gl": "in", "hl": "en"},
        timeout=15,
    )
    resp.raise_for_status()
    return resp.json()


def extract_phone(text: str) -> str:
    """Pull first Indian phone number found in a string."""
    pattern = r"(?:\+91[-\s]?)?[6-9]\d{9}"
    match = re.search(pattern, text.replace(" ", "").replace("-", ""))
    return match.group(0) if match else ""


def classify_link(url: str) -> str:
    """Return the referral key for a URL, or '' if it's not a referral domain."""
    for domain, key in REFERRAL_DOMAINS.items():
        if domain in url:
            return key
    return ""


def is_likely_official(url: str, name: str) -> bool:
    """Heuristic: does the URL look like the restaurant's own site?"""
    if not url:
        return False
    for domain in REFERRAL_DOMAINS:
        if domain in url:
            return False
    skip = ["google.", "facebook.", "instagram.", "twitter.", "youtube.",
            "linkedin.", "wikipedia.", "maps.app.goo"]
    return not any(s in url for s in skip)


def parse_results(data: dict, name: str) -> dict:
    """
    Extract structured lead info from a Serper response dict.
    Checks: knowledgeGraph, organic results, answerBox, places.
    """
    row = {f: "" for f in OUTPUT_FIELDS}
    row["name"] = name
    row["city_hint"] = CITY_HINT
    other_links = []

    # ── Knowledge Graph (often has official site + phone) ──
    kg = data.get("knowledgeGraph", {})
    if kg:
        row["official_website"] = row["official_website"] or kg.get("website", "")
        row["phone"]            = row["phone"]            or kg.get("phoneNumber", "")
        row["google_maps"]      = row["google_maps"]      or kg.get("maps", "")

    # ── Places panel ──
    for place in data.get("places", []):
        row["phone"] = row["phone"] or place.get("phoneNumber", "")
        row["google_maps"] = row["google_maps"] or place.get("cid", "")

    # ── Organic results ──
    for result in data.get("organic", []):
        url     = result.get("link", "")
        snippet = result.get("snippet", "")
        title   = result.get("title", "")

        # First result title/snippet for context
        if not row["search_title"]:
            row["search_title"]   = title
            row["search_snippet"] = snippet[:200]

        key = classify_link(url)
        if key:
            if not row[key]:          # keep first (most relevant) link per platform
                row[key] = url
        elif is_likely_official(url, name):
            if not row["official_website"]:
                row["official_website"] = url
        else:
            if url not in other_links and url:
                other_links.append(url)

        # Try to scrape phone from snippet if still missing
        if not row["phone"]:
            row["phone"] = extract_phone(snippet)

    # ── Answer box ──
    answer = data.get("answerBox", {})
    if not row["phone"]:
        row["phone"] = extract_phone(answer.get("answer", "") + answer.get("snippet", ""))

    row["other_links"] = " | ".join(other_links[:5])  # cap at 5 misc links
    return row


def load_names(filepath: str) -> list[dict]:
    """Read CSV, skip blank names, deduplicate."""
    names = []
    seen  = set()
    with open(filepath, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            name = row.get("name", "").strip()
            if not name or name in seen:
                continue
            # Skip clearly non-useful names
            if re.fullmatch(r"(Canteen|Mess\s*[-–]?\s*\d*)", name, re.IGNORECASE):
                print(f"  ⚠  Skipping generic name: '{name}'")
                continue
            seen.add(name)
            # Carry over any data already in OSM export
            names.append({
                "name":    name,
                "website": row.get("website", "").strip(),
                "phone":   row.get("phone", "").strip(),
            })
    return names


# ── Main ─────────────────────────────────────────────────────────────────────

def main():
    if SERPER_API_KEY == "5aaf23f2e9f016e38bb99da0f136d5f9f66e0f58":
        print("❌  Set your SERPER_API_KEY env variable before running.")
        print("    export SERPER_API_KEY='5aaf23f2e9f016e38bb99da0f136d5f9f66e0f58'")
        return

    entries = load_names(INPUT_FILE)
    print(f"✅  Loaded {len(entries)} unique restaurant names from {INPUT_FILE}")

    results = []

    for i, entry in enumerate(entries, 1):
        name  = entry["name"]
        query = f"{name} restaurant {CITY_HINT}"
        print(f"[{i}/{len(entries)}] Searching: {query}")

        try:
            data = serper_search(query)
            row  = parse_results(data, name)

            # Prefer data already in OSM export over scraped data
            if entry["website"]:
                row["official_website"] = entry["website"]
            if entry["phone"]:
                row["phone"] = entry["phone"]

            results.append(row)

            # Show quick summary
            found = [k for k in ["official_website","phone","zomato","swiggy","magicpin"] if row[k]]
            print(f"   → Found: {', '.join(found) if found else 'nothing useful'}")

        except requests.HTTPError as e:
            print(f"   ✗ HTTP error: {e}")
            results.append({"name": name, "city_hint": CITY_HINT,
                            "search_snippet": f"ERROR: {e}"})
        except Exception as e:
            print(f"   ✗ Error: {e}")
            results.append({"name": name, "city_hint": CITY_HINT,
                            "search_snippet": f"ERROR: {e}"})

        time.sleep(DELAY_SECONDS)

    # ── Write output ──
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(results)

    total     = len(results)
    with_site = sum(1 for r in results if r.get("official_website"))
    with_ph   = sum(1 for r in results if r.get("phone"))
    with_zom  = sum(1 for r in results if r.get("zomato"))
    with_swi  = sum(1 for r in results if r.get("swiggy"))

    print(f"\n{'─'*50}")
    print(f"✅  Done! {total} restaurants scraped → {OUTPUT_FILE}")
    print(f"   Official websites : {with_site}")
    print(f"   Phone numbers     : {with_ph}")
    print(f"   Zomato links      : {with_zom}")
    print(f"   Swiggy links      : {with_swi}")
    print(f"{'─'*50}")


if __name__ == "__main__":
    main()

  ⚠  Skipping generic name: 'Canteen'
  ⚠  Skipping generic name: 'Mess - 2'
  ⚠  Skipping generic name: 'Canteen'
  ⚠  Skipping generic name: 'Mess - 1'
  ⚠  Skipping generic name: 'Mess - 3'
  ⚠  Skipping generic name: 'Mess - 4'
  ⚠  Skipping generic name: 'Mess - 5'
  ⚠  Skipping generic name: 'Mess - 6'
✅  Loaded 55 unique restaurant names from businesses.csv
[1/55] Searching: Filfora restaurant Bhopal
   ✗ HTTP error: 403 Client Error: Forbidden for url: https://google.serper.dev/search
[2/55] Searching: Manohar Dairy & Restaurant restaurant Bhopal
   ✗ HTTP error: 403 Client Error: Forbidden for url: https://google.serper.dev/search
[3/55] Searching: Shree Krishna Chaat restaurant Bhopal
   ✗ HTTP error: 403 Client Error: Forbidden for url: https://google.serper.dev/search
[4/55] Searching: MANIT Canteen restaurant Bhopal
   ✗ HTTP error: 403 Client Error: Forbidden for url: https://google.serper.dev/search


KeyboardInterrupt: 